# EDA & Cleaning: IWPC Full Cohort (n=6256)

Source: `warfit_learn/iwpc_full_6256_patients.xlsx`, the full International Warfarin Pharmacogenetics Consortium dataset (real patients). Candidate as a **primary modeling dataset**: has genotype (VKORC1, CYP2C9, CYP4F2-relevant columns), clinical/demographic covariates, and the therapeutic dose outcome.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", 100)
DATA_DIR = Path("..").resolve()
DECISIONS_PATH = Path("dataset_decisions.json")

def record_decision(name, verdict, reason, n_rows=None, n_cols=None, saved_to=None):
    decisions = {}
    if DECISIONS_PATH.exists():
        decisions = json.loads(DECISIONS_PATH.read_text())
    decisions[name] = {
        "verdict": verdict,
        "reason": reason,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "saved_to": saved_to,
    }
    DECISIONS_PATH.write_text(json.dumps(decisions, indent=2))
    print(f"Recorded decision for '{name}': {verdict}")


## File inventory: `warfit_learn/` (1 data file plus the `warfit_learn` Python package, source and wheel)

**Used:** `iwpc_full_6256_patients.xlsx`.

**Not used (package internals, not separate data):** `warfit_learn-0.2.1-py3-none-any.whl` and `extracted/` are the `warfit_learn` pip package (source plus estimators/preprocessing code), included for its IWPC-benchmarking methodology, not as a second dataset. It also bundles its own copy of the IWPC cohort at `extracted/warfit_learn/datasets/data/iwpc.pkl`: same shape (6256x68) and same columns as `iwpc_full_6256_patients.xlsx`, values match on spot-check. Treat it as a duplicate of the file already used here, not an independent dataset.

In [ ]:
df = pd.read_excel(DATA_DIR / "usable/warfit_learn/iwpc_full_6256_patients.xlsx")
print(df.shape)
df.head(3)

## Column inventory & missingness

In [ ]:
miss = df.isna().mean().sort_values(ascending=False) * 100
miss.to_frame("pct_missing").head(30)

In [ ]:
# Columns fully or near-fully empty (candidates to drop)
empty_cols = miss[miss > 95].index.tolist()
print(f"{len(empty_cols)} columns >95% missing:")
print(empty_cols)

## Key columns for the project (genotype, dose, INR-related)

In [ ]:
key_terms = ["VKORC1", "CYP2C9", "CYP4F2", "Dose", "INR", "Age", "Weight", "Height",
             "Race", "Ethnicity", "Amiodarone", "Enzyme", "Aspirin", "Smok"]
key_cols = [c for c in df.columns if any(t.lower() in c.lower() for t in key_terms)]
print(len(key_cols), "relevant columns found")
key_cols

In [ ]:
target_col = "Therapeutic Dose of Warfarin"
print(target_col in df.columns)
df[target_col].describe()

## Cleaning
- Drop columns that are >95% missing (mostly unused genotype assays / free-text fields).
- Drop rows with a missing target dose (can't be used for supervised training).
- Keep the PharmGKB ID as a row identifier.

In [ ]:
clean = df.drop(columns=empty_cols)
before = len(clean)
clean = clean.dropna(subset=[target_col])
after = len(clean)
print(f"Dropped {before - after} rows missing target dose. Remaining: {after}")
clean.shape

In [ ]:
out_path = Path("cleaned/iwpc_full_6256_clean.csv")
clean.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())

## Verdict

**KEEP: primary modeling dataset.** Real patients, includes genetic (VKORC1/CYP2C9/CYP4F2-linked) and clinical/demographic predictors plus the therapeutic dose outcome. This is the largest real-patient cohort available and should anchor the supervised dose-prediction model.

In [ ]:
record_decision(
    "iwpc_full_6256",
    verdict="KEEP",
    reason="Primary real-patient dataset: genotype + clinical covariates + therapeutic dose target.",
    n_rows=clean.shape[0], n_cols=clean.shape[1], saved_to=str(out_path),
)